<a href="https://colab.research.google.com/github/mrupendrakushwaha/AI-Powered-Smart-Waste-Management-System-Using-Deep-Learning/blob/main/AI_Powered_Smart_Waste_Management_System_Using_Deep_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [48]:
!pip -q install reportlab

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from datetime import datetime
from zoneinfo import ZoneInfo

from google.colab import drive, files
from google.colab.output import eval_js
from IPython.display import display, Javascript

from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

from base64 import b64decode

from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle
)

from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.enums import TA_CENTER
from reportlab.platypus import Image
from reportlab.lib.units import inch
from reportlab.platypus import PageBreak

print("✅ All Libraries Imported Successfully")

✅ All Libraries Imported Successfully


In [ ]:
drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/Waste_Classification_V3/trashnet.zip"
extract_path = "/content/trashnet"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Dataset Extracted Successfully")

print(os.listdir("/content/trashnet"))

print(os.listdir("/content/trashnet/dataset-resized"))

Mounted at /content/drive
✅ Dataset Extracted Successfully
['dataset-resized']
['cardboard', 'glass', 'trash', 'plastic', 'metal', 'paper']


In [ ]:
dataset_path = "/content/trashnet/dataset-resized"

img_size = (224, 224)
batch_size = 32

train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    validation_split=0.2,
    rotation_range=25,
    zoom_range=0.25,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True
)

train_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    subset="training",
    shuffle=True
)

validation_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

print("Class Labels :", train_generator.class_indices)
print("Training Images :", train_generator.samples)
print("Validation Images :", validation_generator.samples)

Found 2024 images belonging to 6 classes.
Found 503 images belonging to 6 classes.
Class Labels : {'cardboard': 0, 'glass': 1, 'metal': 2, 'paper': 3, 'plastic': 4, 'trash': 5}
Training Images : 2024
Validation Images : 503


In [ ]:
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.4)(x)
x = Dense(256, activation="relu")(x)

output = Dense(
    train_generator.num_classes,
    activation="softmax"
)(x)

model = Model(
    inputs=base_model.input,
    outputs=output
)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "best_waste_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=20,
    callbacks=[
        early_stop,
        reduce_lr,
        checkpoint
    ]
)

print("✅ Model Training Completed Successfully")

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10,5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

loss, accuracy = model.evaluate(validation_generator, verbose=0)

print("="*50)
print("MODEL PERFORMANCE")
print("="*50)
print(f"Validation Accuracy : {accuracy*100:.2f}%")
print(f"Validation Loss     : {loss:.4f}")
print("="*50)

In [ ]:
model.save("AI_Smart_Waste_Classifier_V4.keras")

print("✅ Model Saved Successfully")
print("📁 File Name : AI_Smart_Waste_Classifier_V4.keras")

loss, accuracy = model.evaluate(validation_generator, verbose=0)

print("\nFinal Model Performance")
print(f"Accuracy : {accuracy*100:.2f}%")
print(f"Loss     : {loss:.4f}")
# Optional: Load saved model
# model = tf.keras.models.load_model("AI_Smart_Waste_Classifier_V4.keras")

In [21]:
class_names = [
    "cardboard",
    "glass",
    "metal",
    "paper",
    "plastic",
    "trash"
]

history = []

co2_saved = {
    "cardboard": 1.20,
    "glass": 0.30,
    "metal": 2.50,
    "paper": 1.50,
    "plastic": 0.45,
    "trash": 0.00,
    "unknown": 0.00
}

green_points = {
    "cardboard": 4,
    "glass": 4,
    "metal": 7,
    "paper": 3,
    "plastic": 5,
    "trash": 0,
    "unknown": 0
}

green_score_map = {
    "cardboard": 88,
    "glass": 85,
    "metal": 95,
    "paper": 90,
    "plastic": 80,
    "trash": 20,
    "unknown": 0
}

suggestions = {
    "cardboard": "📦 Put in Cardboard Recycling Bin",
    "glass": "🍾 Put in Glass Recycling Bin",
    "metal": "🥫 Put in Metal Recycling Bin",
    "paper": "📄 Put in Paper Recycling Bin",
    "plastic": "♻️ Put in Plastic Recycling Bin",
    "trash": "🗑️ Put in General Waste Bin",
    "unknown": "⚠️ No Waste Detected. Capture a clear waste object."
}

print("✅ Project Variables Initialized")

✅ Project Variables Initialized


In [35]:
def save_result(img_name, waste_type, confidence):

    green_score = green_score_map[waste_type]

    if green_score >= 80:
        eco_rating = "🟢 Excellent"
    elif green_score >= 60:
        eco_rating = "🟡 Good"
    elif green_score >= 40:
        eco_rating = "🟠 Average"
    else:
        eco_rating = "🔴 Poor"

    result = {
        "Date & Time": datetime.now(
            ZoneInfo("Asia/Kolkata")
        ).strftime("%d-%m-%Y %I:%M:%S %p"),

        "Image": os.path.basename(img_name),        "Image Path": img_name,   # 👈 Ye line add kakaro
        "Waste": waste_type,
        "Confidence": round(confidence, 2),
        "CO2 Saved (kg)": co2_saved[waste_type],
        "Green Points": green_points[waste_type],
        "Green Score": green_score,
        "Eco Rating": eco_rating
    }

    history.append(result)

    df = pd.DataFrame(history)
    df.to_csv("Waste_Report.csv", index=False)

    print("✅ Result Saved Successfully")

In [23]:
def predict_image(img_path):

    img = image.load_img(img_path, target_size=(224,224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = tf.keras.applications.efficientnet.preprocess_input(img_array)

    prediction = model.predict(img_array, verbose=0)

    confidence = float(np.max(prediction) * 100)
    predicted_class = class_names[np.argmax(prediction)]

    if confidence < 60:
        predicted_class = "unknown"

    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{predicted_class.upper()} ({confidence:.2f}%)")
    plt.show()

    save_result(
        img_path,          # Full path
        predicted_class,
        confidence
    )

    )

    print("\n==============================")
    print("🗑️ AI SMART WASTE REPORT")
    print("==============================")

    print(f"📂 Image          : {os.path.basename(img_path)}")
    print(f"♻️ Waste Type     : {predicted_class.upper()}")
    print(f"🎯 Confidence     : {confidence:.2f}%")
    print(f"🕒 Date & Time    : {datetime.now(ZoneInfo('Asia/Kolkata')).strftime('%d-%m-%Y %I:%M:%S %p')}")
    print(f"🌱 CO2 Saved      : {co2_saved[predicted_class]} kg")
    print(f"⭐ Green Points   : {green_points[predicted_class]}")
    print(f"🌿 Green Score    : {green_score_map[predicted_class]}/100")

    print(f"💡 Suggestion     : {suggestions[predicted_class]}")

    if predicted_class == "unknown":
        print("❌ Status         : No Waste Detected")
    else:
        print("✅ Status         : Waste Detected")

    return predicted_class, confidence

In [36]:
def take_photo(filename="capture.jpg", quality=0.9):

    js = Javascript('''

    async function takePhoto(quality){

      const div = document.createElement('div');
      const button = document.createElement('button');
      button.textContent = '📷 Capture';

      div.appendChild(button);

      const video = document.createElement('video');
      video.style.display = 'block';

      const stream = await navigator.mediaDevices.getUserMedia({
        video: {
          facingMode: "environment"
        }
      });

      document.body.appendChild(div);
      div.appendChild(video);

      video.srcObject = stream;
      await video.play();

      await new Promise(resolve => button.onclick = resolve);

      const canvas = document.createElement('canvas');

      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;

      canvas.getContext('2d').drawImage(video, 0, 0);

      stream.getTracks().forEach(track => track.stop());

      div.remove();

      return canvas.toDataURL('image/jpeg', quality);
    }

    ''')

    display(js)

    data = eval_js(f"takePhoto({quality})")

    binary = b64decode(data.split(',')[1])

    with open(filename, 'wb') as f:
        f.write(binary)

    return filename

In [37]:
def show_dashboard():

    if len(history) == 0:
        print("❌ No Data Available")
        return

    df = pd.DataFrame(history)

    counts = df["Waste"].value_counts()

    print("\n==============================")
    print("📊 AI SMART WASTE DASHBOARD")
    print("==============================")

    plt.figure(figsize=(6,6))
    plt.pie(
        counts.values,
        labels=counts.index,
        autopct="%1.1f%%",
        startangle=90
    )
    plt.title("Waste Distribution")
    plt.savefig("pie_chart.png", dpi=300, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(7,4))
    plt.bar(counts.index, counts.values)
    plt.title("Waste Count")
    plt.xlabel("Waste Type")
    plt.ylabel("Images")
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.savefig("bar_chart.png", dpi=300, bbox_inches="tight")
    plt.show()

    total_images = len(df)
    total_co2 = df["CO2 Saved (kg)"].sum()
    total_points = df["Green Points"].sum()

    recyclable = df[~df["Waste"].isin(["trash", "unknown"])]
    recycling_rate = (len(recyclable) / total_images) * 100

    print(f"📷 Total Images      : {total_images}")
    print(f"🌱 Total CO₂ Saved   : {total_co2:.2f} kg")
    print(f"⭐ Green Points      : {total_points}")
    print(f"♻ Recycling Rate    : {recycling_rate:.1f}%")

    if recycling_rate >= 80:
        print("🌿 Overall Rating    : Excellent")
    elif recycling_rate >= 60:
        print("🌿 Overall Rating    : Good")
    elif recycling_rate >= 40:
        print("🌿 Overall Rating    : Average")
    else:
        print("🌿 Overall Rating    : Needs Improvement")

In [54]:
def generate_pdf():

    if len(history) == 0:
        print("❌ No Data Available")
        return

    df = pd.DataFrame(history)

    pdf = SimpleDocTemplate("AI_Smart_Waste_Report.pdf")
    styles = getSampleStyleSheet()

    title = styles["Heading1"]
    title.alignment = TA_CENTER

    story = []

    story.append(Paragraph(
    "<font size=20 color='darkgreen'><b>AI SMART WASTE MANAGEMENT SYSTEM</b></font>",
    title))
    story.append(Spacer(1, 15))

    story.append(
        Paragraph(
            "<b>Generated On:</b> " +
            datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%d-%m-%Y %I:%M:%S %p"),
            styles["Normal"]
        )
    )

    story.append(Spacer(1, 12))

    total_images = len(df)
    total_co2 = df["CO2 Saved (kg)"].sum()
    total_points = df["Green Points"].sum()

    story.append(Paragraph(f"<b>Total Images:</b> {total_images}", styles["Normal"]))
    story.append(Paragraph(f"<b>Total CO2 Saved:</b> {total_co2:.2f} kg", styles["Normal"]))
    story.append(Paragraph(f"<b>Total Green Points:</b> {total_points}", styles["Normal"]))

    story.append(Spacer(1, 15))

    data = [[
        "Image",
        "Waste",
        "Confidence",
        "CO2 Saved",
        "Green Score"
    ]]

    for _, row in df.iterrows():

        if os.path.exists(row["Image Path"]):
            img = Image(
                row["Image Path"],
                width=1.0*inch,
                height=1.0*inch
            )
        else:
            img = Paragraph("No Image", styles["Normal"])

        data.append([
            img,
            row["Waste"].upper(),
            f"{row['Confidence']:.2f}%",
            f"{row['CO2 Saved (kg)']:.2f}",
            row["Green Score"]
        ])

    table = Table(
data,
colWidths=[140,55,70,70,70]
)

    table.setStyle(TableStyle([

        ("BACKGROUND",(0,0),(-1,0),colors.green),
        ("TEXTCOLOR",(0,0),(-1,0),colors.white),

        ("GRID",(0,0),(-1,-1),1,colors.black),

        ("ALIGN",(0,0),(-1,-1),"CENTER"),

        ("BACKGROUND",(0,1),(-1,-1),colors.beige)

    ]))
    story.append(table)

    story.append(PageBreak())

    story.append(Paragraph("<b>Dashboard Summary</b>", styles["Heading2"]))
    story.append(Spacer(1,10))
    if os.path.exists("pie_chart.png"):
        story.append(Image("pie_chart.png", width=4.5*inch, height=4.5*inch))
        story.append(Spacer(1, 10))

    if os.path.exists("bar_chart.png"):
        story.append(Image("bar_chart.png", width=6*inch, height=3*inch))
        story.append(Spacer(1, 15))


    story.append(Spacer(1,20))

    story.append(
        Paragraph(
            "<b>Project Outcome</b>",
            styles["Heading2"]
        )
    )

    story.append(
        Paragraph(
            "This AI-powered system classifies waste using Deep Learning "
            "and promotes sustainable waste management by estimating "
            "environmental impact through CO2 savings and Green Score.",
            styles["Normal"]
        )
    )

    pdf.build(story)

    print("✅ PDF Generated Successfully")

    files.download("AI_Smart_Waste_Report.pdf")

In [39]:
def reset_data():

    global history

    history.clear()

    if os.path.exists("Waste_Report.csv"):
        os.remove("Waste_Report.csv")

    print("\n==============================")
    print("🗑️ AI SMART WASTE CLASSIFIER")
    print("==============================")
    print("✅ History Cleared")
    print("✅ Waste_Report.csv Deleted")
    print("✅ Dashboard Reset Successfully")
    print("==============================")

In [57]:
while True:

    print("\n==============================")
    print("🗑️ AI SMART WASTE CLASSIFIER")
    print("==============================")
    print("1. Camera")
    print("2. Gallery")
    print("3. Dashboard")
    print("4. Generate PDF")
    print("5. Reset Data")
    print("6. Exit")

    choice = input("\nSelect (1-6): ")

    if choice == "1":

        img_path = take_photo()

        predicted_class, confidence = predict_image(img_path)

    elif choice == "2":

        uploaded = files.upload()

        for img_path in uploaded.keys():

            predicted_class, confidence = predict_image(img_path)

    elif choice == "3":

        show_dashboard()

    elif choice == "4":

        generate_pdf()

    elif choice == "5":

        confirm = input("Are you sure? (y/n): ")

        if confirm.lower() == "y":
            reset_data()
        else:
            print("❌ Reset Cancelled")

    elif choice == "6":

        print("\n👋 Thank You")
        print("AI SMART WASTE MANAGEMENT SYSTEM CLOSED")
        break

    else:

        print("❌ Invalid Choice! Please Try Again.")


🗑️ AI SMART WASTE CLASSIFIER
1. Camera
2. Gallery
3. Dashboard
4. Generate PDF
5. Reset Data
6. Exit

Select (1-6): 6

👋 Thank You
AI SMART WASTE MANAGEMENT SYSTEM CLOSED
